# Toxic Content Detector

**Model:** unitary/toxic-bert | **Size:** 110MB | **Product:** prod-ovjkkkdqpu3je

A BERT-based model fine-tuned on the Jigsaw Toxic Comment Classification dataset for multi-label toxic content detection. Classifies text across categories including toxic, severe toxic, obscene, threat, insult, and identity hate — essential for automated content moderation.

## Use Cases
- Community platform comment moderation
- Social media content policy enforcement
- User-generated content screening before publication
- Online gaming chat toxicity filtering

In [ ]:
import boto3
import sagemaker
from sagemaker import ModelPackage

region = boto3.Session().region_name
role = sagemaker.get_execution_role()
sm_client = boto3.client('sagemaker', region_name=region)

print(f'Region: {region}')
print(f'Role: {role}')

In [ ]:
# Replace with your actual Model Package ARN from AWS Marketplace
model_package_arn = 'arn:aws:sagemaker:REGION:ACCOUNT:model-package/MODEL_PACKAGE_NAME'

# Validate ARN before deploying
if 'REGION' in model_package_arn or 'ACCOUNT' in model_package_arn or 'MODEL_PACKAGE_NAME' in model_package_arn:
    raise ValueError(
        'model_package_arn contains placeholder values. '
        'Subscribe to the model on AWS Marketplace and replace with the actual ARN.'
    )

endpoint_name = 'toxic-content-detector'
instance_type = 'ml.m5.xlarge'

try:
    model = ModelPackage(
        role=role,
        model_package_arn=model_package_arn,
        sagemaker_session=sagemaker.Session()
    )
    predictor = model.deploy(
        initial_instance_count=1,
        instance_type=instance_type,
        endpoint_name=endpoint_name
    )
    print(f'Endpoint deployed: {endpoint_name}')
except Exception as e:
    print(f'Deployment failed: {e}')
    raise

## Step 2: Run Inference

Send text for multi-label toxicity classification. Returns scores for each toxicity category.

In [ ]:
import json

runtime = boto3.client('sagemaker-runtime', region_name=region)

# Sample comments for moderation screening
texts = [
    'Thank you for sharing this helpful tutorial, I learned a lot!',
    'This product is absolutely terrible and a complete waste of money.',
    'Great discussion everyone, really appreciate the diverse perspectives here.',
]

for text in texts:
    payload = json.dumps({'inputs': text})
    try:
        response = runtime.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=payload
        )
        result_raw = response['Body'].read().decode('utf-8')
        try:
            result = json.loads(result_raw)
            print(f'Text: "{text[:60]}"')
            print(f'Toxicity scores: {result}\n')
        except json.JSONDecodeError:
            print(f'Raw response: {result_raw}')
    except Exception as e:
        print(f'Inference failed: {e}')
        raise

In [ ]:
# Cleanup - delete the endpoint to avoid ongoing charges
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print(f'Endpoint {endpoint_name} deleted.')
except Exception as e:
    print(f'Cleanup failed: {e}')